In [1]:
%cd ..

/Users/faizanfaisal/Laptop_Data/Davis_Courses/SE/project/notjs/Not.js


In [2]:
import pandas as pd
import os
import json
from pathlib import Path


In [3]:
# Build request_id -> URL
df = pd.read_json("output/booking.com/request.json", lines=True)
# If the column is named requestId instead of request_id, use that
id_to_url = dict(zip(df["request_id"], df["http_req"]))

# For response/7123.1072.txt
request_id = "7123.5"
js_url = id_to_url.get(request_id)  # JavaScript URL for that file
js_url

'https://cf.bstatic.com/psb/capla/static/css/client.296f9939.css'

In [4]:
base_dir = Path("output")

dfs = []

for website_dir in base_dir.iterdir():
    if website_dir.is_dir():
        file_path = website_dir / "features.xlsx"
        if file_path.exists():
            df = pd.read_excel(file_path)
            df["website"] = website_dir.name  # optional: keep track of source
            dfs.append(df)

combined_df = pd.concat(dfs, ignore_index=True)

print(combined_df)

      Unnamed: 0                                        script_name  \
0             18                 https://www.hp.com/us-en/home.html   
1             20                 https://www.hp.com/us-en/home.html   
2             22  https://www.googletagmanager.com/gtm.js?id=GTM...   
3             24  https://www.googletagmanager.com/gtm.js?id=GTM...   
4             26  https://www.googletagmanager.com/gtm.js?id=GTM...   
...          ...                                                ...   
6013       24847  https://px.mountain.com/st?ga_tracking_id=G-LX...   
6014       24855  https://px.mountain.com/st?ga_tracking_id=G-LX...   
6015       24859                         https://gs.mountain.com/gs   
6016       24863                         https://gs.mountain.com/gs   
6017       24867  https://px.mountain.com/st?ga_tracking_id=G-LX...   

            method_name  label  is_mixed  num_req_sent  num_nodes  num_edges  \
0               @309@63      0         0             0        495  

In [5]:
js_df = combined_df[combined_df["script_name"].str.contains(r"\.js", na=False)]

In [6]:
js_df

,Unnamed: 0,script_name,method_name,label,is_mixed,num_req_sent,num_nodes,num_edges,nodes_div_by_edges,edges_div_by_nodes,...,removeEventListener,sendBeacon,has_fingerprinting_eventlistner,descendant_of_fingerprinting,ascendant_of_fingerprinting,num_local,num_closure,num_global,num_script,website
2,22,https://www.googletagmanager.com/gtm.js?id=GTM...,ms@483@53,0,0,0,495,2053,0.241111,4.147475,...,0,0,0,0,5,0,0,0,0,hp.com
3,24,https://www.googletagmanager.com/gtm.js?id=GTM...,ks@476@1152,0,0,0,495,2053,0.241111,4.147475,...,0,0,0,0,5,0,0,0,0,hp.com
4,26,https://www.googletagmanager.com/gtm.js?id=GTM...,js@476@1022,0,0,0,495,2053,0.241111,4.147475,...,0,0,0,0,5,0,0,0,0,hp.com
5,28,https://www.googletagmanager.com/gtm.js?id=GTM...,@824@229,0,0,0,495,2053,0.241111,4.147475,...,0,0,0,0,5,0,0,0,0,hp.com
6,30,https://www.googletagmanager.com/gtm.js?id=GTM...,@826@237,0,0,0,495,2053,0.241111,4.147475,...,0,0,0,0,5,0,0,0,0,hp.com
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6000,24095,https://bat.bing.com/bat.js,UET._push@0@18667,1,0,4,3925,17820,0.220258,4.540127,...,0,0,0,0,0,0,0,0,0,hubspot.com
6001,24623,https://bat.bing.com/bat.js,UET.fireBeacon@0@43326,1,0,2,3925,17820,0.220258,4.540127,...,0,0,0,27,0,0,0,0,0,hubspot.com
6002,24639,https://bat.bing.com/bat.js,UET._push@0@16964,1,0,2,3925,17820,0.220258,4.540127,...,0,0,0,27,0,0,0,0,0,hubspot.com
6003,24703,https://bat.bing.com/bat.js,UET.checkuetHostdocumentload@0@7903,1,0,2,3925,17820,0.220258,4.540127,...,0,2,0,27,0,0,0,0,0,hubspot.com


In [7]:
# cache for already loaded json mappings
request_maps = {}

def get_js_path(row):
    website = row["website"]
    script = row["script_name"]

    json_path = f"./output/{website}/request_id.json"

    # load mapping once per website
    if website not in request_maps:
        if os.path.exists(json_path):
            with open(json_path, "r") as f:
                request_maps[website] = json.load(f)
        else:
            request_maps[website] = {}

    mapping = request_maps[website]

    # get request id
    req_id = mapping.get(script)

    if req_id is None:
        return None

    return f"./output/{website}/response/{req_id}.txt"


# create new column
js_df["js_file_path"] = js_df.apply(get_js_path, axis=1)

In [12]:
df_with_js_file_path = js_df[js_df.js_file_path.notna()]
df_with_js_file_path

,Unnamed: 0,script_name,method_name,label,is_mixed,num_req_sent,num_nodes,num_edges,nodes_div_by_edges,edges_div_by_nodes,...,sendBeacon,has_fingerprinting_eventlistner,descendant_of_fingerprinting,ascendant_of_fingerprinting,num_local,num_closure,num_global,num_script,website,js_file_path
2,22,https://www.googletagmanager.com/gtm.js?id=GTM...,ms@483@53,0,0,0,495,2053,0.241111,4.147475,...,0,0,0,5,0,0,0,0,hp.com,./output/hp.com/response/5232.12.txt
3,24,https://www.googletagmanager.com/gtm.js?id=GTM...,ks@476@1152,0,0,0,495,2053,0.241111,4.147475,...,0,0,0,5,0,0,0,0,hp.com,./output/hp.com/response/5232.12.txt
4,26,https://www.googletagmanager.com/gtm.js?id=GTM...,js@476@1022,0,0,0,495,2053,0.241111,4.147475,...,0,0,0,5,0,0,0,0,hp.com,./output/hp.com/response/5232.12.txt
5,28,https://www.googletagmanager.com/gtm.js?id=GTM...,@824@229,0,0,0,495,2053,0.241111,4.147475,...,0,0,0,5,0,0,0,0,hp.com,./output/hp.com/response/5232.12.txt
6,30,https://www.googletagmanager.com/gtm.js?id=GTM...,@826@237,0,0,0,495,2053,0.241111,4.147475,...,0,0,0,5,0,0,0,0,hp.com,./output/hp.com/response/5232.12.txt
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6000,24095,https://bat.bing.com/bat.js,UET._push@0@18667,1,0,4,3925,17820,0.220258,4.540127,...,0,0,0,0,0,0,0,0,hubspot.com,./output/hubspot.com/response/3419.1259.txt
6001,24623,https://bat.bing.com/bat.js,UET.fireBeacon@0@43326,1,0,2,3925,17820,0.220258,4.540127,...,0,0,27,0,0,0,0,0,hubspot.com,./output/hubspot.com/response/3419.1259.txt
6002,24639,https://bat.bing.com/bat.js,UET._push@0@16964,1,0,2,3925,17820,0.220258,4.540127,...,0,0,27,0,0,0,0,0,hubspot.com,./output/hubspot.com/response/3419.1259.txt
6003,24703,https://bat.bing.com/bat.js,UET.checkuetHostdocumentload@0@7903,1,0,2,3925,17820,0.220258,4.540127,...,2,0,27,0,0,0,0,0,hubspot.com,./output/hubspot.com/response/3419.1259.txt


In [5]:
import sys
from pathlib import Path

# Ensure project root is on path for surrogate_2 import (notebook may run from pipeline/)
_root = Path.cwd()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from surrogate_2.replace_function_call import extract_call_snippet
from surrogate_2.balance_parentheses import find_function_definition


def build_script_path_cache(base_dir: Path) -> dict:
    """Build (website, script_url) -> path to response .txt file from each site's request.json."""
    cache = {}
    for website_dir in base_dir.iterdir():
        if not website_dir.is_dir():
            continue
        request_file = website_dir / "request.json"
        if not request_file.exists():
            continue
        try:
            req_df = pd.read_json(request_file, lines=True)
        except Exception:
            continue
        for i in req_df.index:
            url = req_df["http_req"].iloc[i]
            req_id = req_df["request_id"].iloc[i]
            path = website_dir / "response" / f"{req_id}.txt"
            if path.exists():
                cache[(website_dir.name, url)] = path
    return cache


def extract_method_code(script_path, method_name):
    """
    Extract the function call code at method_name (fn@line@col, 0-based) from script at script_path.
    Returns the call text string or None if not found.
    """
    if script_path is None:
        return None
    path = Path(script_path)
    if not path.exists():
        return None
    parts = str(method_name).split("@")
    if len(parts) < 3:
        return None
    try:
        line_0 = int(parts[1])
        col_0 = int(parts[2])
    except ValueError:
        return None
    # extract_call_snippet expects 1-based line/column
    line_1 = line_0 + 1
    col_1 = col_0 + 1
    try:
        with open(path, "r", encoding="utf-8", errors="replace") as f:
            code_lines = f.readlines()
    except Exception:
        return None
    call_text, _ = extract_call_snippet(code_lines, line_1, col_1)
    return call_text


# Build cache: (website, script_name) -> path to script source
path_cache = build_script_path_cache(base_dir)


def get_code_for_row(row):
    key = (row["website"], row["script_name"])
    path = path_cache.get(key)
    return extract_method_code(path, row["method_name"])


def extract_method_definition(script_path, callee_name):
    """
    Find and return the source code of the function definition for callee_name in the script.
    Returns the full function (signature + body) or None if not found.
    """
    if not callee_name or not str(callee_name).strip():
        return None
    if script_path is None:
        return None
    path = Path(script_path)
    if not path.exists():
        return None
    try:
        code = path.read_text(encoding="utf-8", errors="replace")
    except Exception:
        return None
    return find_function_definition(code, str(callee_name).strip())


def get_definition_for_row(row):
    key = (row["website"], row["script_name"])
    path = path_cache.get(key)
    callee_name = str(row["method_name"]).split("@")[0]
    return extract_method_definition(path, callee_name)


js_df["method_code"] = js_df.apply(get_code_for_row, axis=1)
js_df["method_definition"] = js_df.apply(get_definition_for_row, axis=1)
js_df

,Unnamed: 0,script_name,method_name,label,is_mixed,num_req_sent,num_nodes,num_edges,nodes_div_by_edges,edges_div_by_nodes,...,has_fingerprinting_eventlistner,descendant_of_fingerprinting,ascendant_of_fingerprinting,num_local,num_closure,num_global,num_script,website,method_code,method_definition
2,22,https://www.googletagmanager.com/gtm.js?id=GTM...,ms@483@53,0,0,0,495,2053,0.241111,4.147475,...,0,0,5,0,0,0,0,hp.com,"(c.domain=""auto"")",ms=function(a){return a&&hb(6)?(Array.isArray(...
3,24,https://www.googletagmanager.com/gtm.js?id=GTM...,ks@476@1152,0,0,0,495,2053,0.241111,4.147475,...,0,0,5,0,0,0,0,hp.com,NaN,"function js(a){return a.origin!==""null""};funct..."
4,26,https://www.googletagmanager.com/gtm.js?id=GTM...,js@476@1022,0,0,0,495,2053,0.241111,4.147475,...,0,0,5,0,0,0,0,hp.com,NaN,function hs(a){if(hb(20)&&ds.includes(a)){var ...
5,28,https://www.googletagmanager.com/gtm.js?id=GTM...,@824@229,0,0,0,495,2053,0.241111,4.147475,...,0,0,5,0,0,0,0,hp.com,NaN,NaN
6,30,https://www.googletagmanager.com/gtm.js?id=GTM...,@826@237,0,0,0,495,2053,0.241111,4.147475,...,0,0,5,0,0,0,0,hp.com,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6000,24095,https://bat.bing.com/bat.js,UET._push@0@18667,1,0,4,3925,17820,0.220258,4.540127,...,0,0,0,0,0,0,0,hubspot.com,"this.fireConsentPing(""update"")",NaN
6001,24623,https://bat.bing.com/bat.js,UET.fireBeacon@0@43326,1,0,2,3925,17820,0.220258,4.540127,...,0,27,0,0,0,0,0,hubspot.com,this.fireBeaconImg(n),NaN
6002,24639,https://bat.bing.com/bat.js,UET._push@0@16964,1,0,2,3925,17820,0.220258,4.540127,...,0,27,0,0,0,0,0,hubspot.com,"this.evt(o,i,t,e[2])",NaN
6003,24703,https://bat.bing.com/bat.js,UET.checkuetHostdocumentload@0@7903,1,0,2,3925,17820,0.220258,4.540127,...,0,27,0,0,0,0,0,hubspot.com,e._push(e.eventPushQueue[t]),NaN


In [6]:
for i in js_df['script_name']:
    print(i)

https://www.googletagmanager.com/gtm.js?id=GTM-P9TRBRZ
https://www.googletagmanager.com/gtm.js?id=GTM-P9TRBRZ
https://www.googletagmanager.com/gtm.js?id=GTM-P9TRBRZ
https://www.googletagmanager.com/gtm.js?id=GTM-P9TRBRZ
https://www.googletagmanager.com/gtm.js?id=GTM-P9TRBRZ
https://www.googletagmanager.com/gtm.js?id=GTM-P9TRBRZ
https://www.googletagmanager.com/gtm.js?id=GTM-P9TRBRZ
https://cdn.dynamicyield.com/api/8784964/api_static.js
https://cdn.dynamicyield.com/api/8784964/api_static.js
https://cdn.dynamicyield.com/api/8784964/api_static.js
https://cdn.dynamicyield.com/api/8784964/api_static.js
https://cdn.dynamicyield.com/api/8784964/api_static.js
https://cdn.dynamicyield.com/api/8784964/api_static.js
https://cdn.dynamicyield.com/api/8784964/api_static.js
https://cdn.dynamicyield.com/api/8784964/api_static.js
https://cdn.dynamicyield.com/api/8784964/api_static.js
https://cdn.dynamicyield.com/api/8784964/api_static.js
https://cdn.dynamicyield.com/api/8784964/api_static.js
https://cd

In [6]:
js_df.to_csv("js_df.csv", index=False)

In [7]:
df = js_df.copy()

In [8]:
# Find the start and end positions
start = df.columns.get_loc("is_mixed")
end = df.columns.get_loc("num_script")

# Columns to rename
cols_to_rename = df.columns[start:end+1]

# Create new names
new_names = {col: f"Feature {i+1}" for i, col in enumerate(cols_to_rename)}

# Rename
df = df.rename(columns=new_names)

In [9]:
df

,Unnamed: 0,script_name,method_name,label,Feature 1,Feature 2,Feature 3,Feature 4,Feature 5,Feature 6,...,Feature 37,Feature 38,Feature 39,Feature 40,Feature 41,Feature 42,Feature 43,website,method_code,method_definition
2,22,https://www.googletagmanager.com/gtm.js?id=GTM...,ms@483@53,0,0,0,495,2053,0.241111,4.147475,...,0,0,5,0,0,0,0,hp.com,"(c.domain=""auto"")",ms=function(a){return a&&hb(6)?(Array.isArray(...
3,24,https://www.googletagmanager.com/gtm.js?id=GTM...,ks@476@1152,0,0,0,495,2053,0.241111,4.147475,...,0,0,5,0,0,0,0,hp.com,NaN,"function js(a){return a.origin!==""null""};funct..."
4,26,https://www.googletagmanager.com/gtm.js?id=GTM...,js@476@1022,0,0,0,495,2053,0.241111,4.147475,...,0,0,5,0,0,0,0,hp.com,NaN,function hs(a){if(hb(20)&&ds.includes(a)){var ...
5,28,https://www.googletagmanager.com/gtm.js?id=GTM...,@824@229,0,0,0,495,2053,0.241111,4.147475,...,0,0,5,0,0,0,0,hp.com,NaN,NaN
6,30,https://www.googletagmanager.com/gtm.js?id=GTM...,@826@237,0,0,0,495,2053,0.241111,4.147475,...,0,0,5,0,0,0,0,hp.com,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6000,24095,https://bat.bing.com/bat.js,UET._push@0@18667,1,0,4,3925,17820,0.220258,4.540127,...,0,0,0,0,0,0,0,hubspot.com,"this.fireConsentPing(""update"")",NaN
6001,24623,https://bat.bing.com/bat.js,UET.fireBeacon@0@43326,1,0,2,3925,17820,0.220258,4.540127,...,0,27,0,0,0,0,0,hubspot.com,this.fireBeaconImg(n),NaN
6002,24639,https://bat.bing.com/bat.js,UET._push@0@16964,1,0,2,3925,17820,0.220258,4.540127,...,0,27,0,0,0,0,0,hubspot.com,"this.evt(o,i,t,e[2])",NaN
6003,24703,https://bat.bing.com/bat.js,UET.checkuetHostdocumentload@0@7903,1,0,2,3925,17820,0.220258,4.540127,...,0,27,0,0,0,0,0,hubspot.com,e._push(e.eventPushQueue[t]),NaN


In [10]:
df.iloc[:, :-3].to_csv("df_pipeline.csv", index=False)

In [11]:
from joblib import load
from classifier import read_dataset, generateConfig

#################### Downloading Dataset and Models ####################
data/notjs.joblib already exists, skipping.
data/notjs.csv already exists, skipping.
data/notjs_ablation_1.joblib already exists, skipping.
data/notjs_ablation_2.joblib already exists, skipping.
data/performance_output.json already exists, skipping.
data/performance_output_surr.json already exists, skipping.
data/surrogate-generate-cdf.csv already exists, skipping.
data/test_coverage.csv already exists, skipping.
data/test_obfuscation.csv already exists, skipping.
test-training split done
#################### Table 3 ####################
            Tracking Functions  Non-tracking Functions    Total
Training                408429                  674724  1083153
Validation              135590                  225462   361052
Testing                 136045                  225007   361052
#################### Model Loaded Successfully ####################
#################### Table 4 - Standard 5.1 ###############

In [12]:
# Load dataset and model
model_path = "/Users/faizanfaisal/Laptop_Data/Davis_Courses/SE/project/notjs/Not.js/data/notjs.joblib"
dataset = read_dataset("df_pipeline.csv")
labels, features, weights = generateConfig(dataset, ['Feature 2'], False, 4)
model = load(model_path)

In [13]:
# Run predictions on the whole dataset (or a slice)
y_pred = model.predict(features)

# Attach predictions back to the DataFrame if you want
dataset_with_pred = dataset.copy()
dataset_with_pred["prediction"] = y_pred

dataset_with_pred

,Unnamed: 0,script_name,method_name,label,Feature 1,Feature 2,Feature 3,Feature 4,Feature 5,Feature 6,...,Feature 35,Feature 36,Feature 37,Feature 38,Feature 39,Feature 40,Feature 41,Feature 42,Feature 43,prediction
0,22,https://www.googletagmanager.com/gtm.js?id=GTM...,ms@483@53,0,0,0,3925,17820,0.220258,4.540127,...,0,0,0,0,0,0,0,0,0,1
1,24,https://www.googletagmanager.com/gtm.js?id=GTM...,ks@476@1152,0,0,0,3925,17820,0.220258,4.540127,...,0,0,0,0,0,0,0,0,0,1
2,26,https://www.googletagmanager.com/gtm.js?id=GTM...,js@476@1022,0,0,0,3925,17820,0.220258,4.540127,...,0,0,0,0,0,0,0,0,0,1
3,28,https://www.googletagmanager.com/gtm.js?id=GTM...,@824@229,0,0,0,3925,17820,0.220258,4.540127,...,0,0,0,0,0,0,0,0,0,1
4,30,https://www.googletagmanager.com/gtm.js?id=GTM...,@826@237,0,0,0,3925,17820,0.220258,4.540127,...,0,0,0,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1836,24095,https://bat.bing.com/bat.js,UET._push@0@18667,1,0,4,3925,17820,0.220258,4.540127,...,0,0,0,0,0,0,0,0,0,1
1837,24623,https://bat.bing.com/bat.js,UET.fireBeacon@0@43326,1,0,2,3925,17820,0.220258,4.540127,...,0,0,0,27,0,0,0,0,0,1
1838,24639,https://bat.bing.com/bat.js,UET._push@0@16964,1,0,2,3925,17820,0.220258,4.540127,...,0,0,0,27,0,0,0,0,0,1
1839,24703,https://bat.bing.com/bat.js,UET.checkuetHostdocumentload@0@7903,1,0,2,3925,17820,0.220258,4.540127,...,0,2,0,27,0,0,0,0,0,1


In [14]:
result = pd.concat([dataset_with_pred, df.iloc[:, -3:]], axis=1)

In [29]:
result.script_name.iloc[0]

'https://www.googletagmanager.com/gtm.js?id=GTM-P9TRBRZ'

In [30]:
result

,Unnamed: 0,script_name,method_name,label,Feature 1,Feature 2,Feature 3,Feature 4,Feature 5,Feature 6,...,Feature 38,Feature 39,Feature 40,Feature 41,Feature 42,Feature 43,prediction,website,method_code,method_definition
0,22.0,https://www.googletagmanager.com/gtm.js?id=GTM...,ms@483@53,0.0,0.0,0.0,3925.0,17820.0,0.220258,4.540127,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,NaN,NaN,NaN
1,24.0,https://www.googletagmanager.com/gtm.js?id=GTM...,ks@476@1152,0.0,0.0,0.0,3925.0,17820.0,0.220258,4.540127,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,NaN,NaN,NaN
2,26.0,https://www.googletagmanager.com/gtm.js?id=GTM...,js@476@1022,0.0,0.0,0.0,3925.0,17820.0,0.220258,4.540127,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,hp.com,"(c.domain=""auto"")",ms=function(a){return a&&hb(6)?(Array.isArray(...
3,28.0,https://www.googletagmanager.com/gtm.js?id=GTM...,@824@229,0.0,0.0,0.0,3925.0,17820.0,0.220258,4.540127,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,hp.com,NaN,"function js(a){return a.origin!==""null""};funct..."
4,30.0,https://www.googletagmanager.com/gtm.js?id=GTM...,@826@237,0.0,0.0,0.0,3925.0,17820.0,0.220258,4.540127,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,hp.com,NaN,function hs(a){if(hb(20)&&ds.includes(a)){var ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,hubspot.com,"this.fireConsentPing(""update"")",NaN
6001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,hubspot.com,this.fireBeaconImg(n),NaN
6002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,hubspot.com,"this.evt(o,i,t,e[2])",NaN
6003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,hubspot.com,e._push(e.eventPushQueue[t]),NaN


In [15]:
temp = result[result["prediction"] == result["label"]]


In [33]:
temp_2 = temp[temp.method_code.notna() & temp.method_definition.notna()]
temp_2



,Unnamed: 0,script_name,method_name,label,Feature 1,Feature 2,Feature 3,Feature 4,Feature 5,Feature 6,...,Feature 38,Feature 39,Feature 40,Feature 41,Feature 42,Feature 43,prediction,website,method_code,method_definition
654,9796.0,https://connect.facebook.net/en_US/fbevents.js,plugin@424@1766,1.0,0.0,2.0,3925.0,17820.0,0.220258,4.540127,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,booking.com,"_0x19f919[_0x19855d(0x0,0x472,0x0,0x4be)](_0x2...","function _0x281135(_0x39f6b5,_0x2b0460,_0x26e0..."
658,9816.0,https://connect.facebook.net/en_US/fbevents.js,@415@4,1.0,0.0,2.0,3925.0,17820.0,0.220258,4.540127,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,booking.com,"function _0x3cba20(_0x19838f,_0x3e2906,_0x86a3...","_0x2e727f=function(_0x401c58,_0x20845b,_0x2b8f..."
664,9840.0,https://connect.facebook.net/en_US/fbevents.js,Oe@412@3493,1.0,0.0,4.0,3925.0,17820.0,0.220258,4.540127,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,booking.com,"_0x4eb5c1['BGmwW'](_0x2e727f,this,void 0x0,voi...",function(_0x209abc){return _0x3e5e47(setTimeou...
669,9984.0,https://snap.licdn.com/li.lms-analytics/insigh...,@0@32316,1.0,0.0,6.0,3925.0,17820.0,0.220258,4.540127,...,0.0,0.0,1.0,2.0,1.0,1.0,1.0,booking.com,return new(_0x42c4c6||(_0x42c4c6=Promise)),"_0x523f75=function(_0x24166a,_0x1b8b51,_0x42c4..."
670,9986.0,https://snap.licdn.com/li.lms-analytics/insigh...,@0@12097,1.0,0.0,2.0,3925.0,17820.0,0.220258,4.540127,...,0.0,0.0,1.0,2.0,1.0,1.0,1.0,booking.com,"function _0x318087(_0x5dbb61,_0xe317df,_0x1d4a...","function _0x2ba606(_0x4ca0c1,_0x35df86,_0x3a96..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1746,22027.0,https://static.ads-twitter.com/uwt.js,2735@0@34276,1.0,0.0,4.0,3925.0,17820.0,0.220258,4.540127,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,shopify.com,"m(o,e)","function(e,t){return e.__proto__=t,e},_(e,t)}f..."
1747,22031.0,https://static.ads-twitter.com/uwt.js,@0@47540,1.0,0.0,4.0,3925.0,17820.0,0.220258,4.540127,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,shopify.com,"d({customData:r,customParams:u,eventName:a,id:...","function Qe(t){var n="""";if(t.tagName===""IMG"")r..."
1748,22035.0,https://static.ads-twitter.com/uwt.js,@0@33973,1.0,0.0,4.0,3925.0,17820.0,0.220258,4.540127,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,shopify.com,"C(r,e.experimentId)","function(t){return this._invoke(e,n,t)})}n?o?o..."
1750,22043.0,https://static.ads-twitter.com/uwt.js,u@0@31050,1.0,0.0,2.0,3925.0,17820.0,0.220258,4.540127,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,shopify.com,"(e,{highFetchPriority:i,expv2:l})","function y(e,t,n,r,o,a,i){try{var l=e[a](i),s=..."


In [34]:
temp_2.iloc[:,-2:].to_csv("temp_2.csv", index=False)

In [35]:
temp_2.script_name.iloc[4]

'https://snap.licdn.com/li.lms-analytics/insight.min.js'

In [37]:
temp_2

,Unnamed: 0,script_name,method_name,label,Feature 1,Feature 2,Feature 3,Feature 4,Feature 5,Feature 6,...,Feature 38,Feature 39,Feature 40,Feature 41,Feature 42,Feature 43,prediction,website,method_code,method_definition
654,9796.0,https://connect.facebook.net/en_US/fbevents.js,plugin@424@1766,1.0,0.0,2.0,3925.0,17820.0,0.220258,4.540127,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,booking.com,"_0x19f919[_0x19855d(0x0,0x472,0x0,0x4be)](_0x2...","function _0x281135(_0x39f6b5,_0x2b0460,_0x26e0..."
658,9816.0,https://connect.facebook.net/en_US/fbevents.js,@415@4,1.0,0.0,2.0,3925.0,17820.0,0.220258,4.540127,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,booking.com,"function _0x3cba20(_0x19838f,_0x3e2906,_0x86a3...","_0x2e727f=function(_0x401c58,_0x20845b,_0x2b8f..."
664,9840.0,https://connect.facebook.net/en_US/fbevents.js,Oe@412@3493,1.0,0.0,4.0,3925.0,17820.0,0.220258,4.540127,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,booking.com,"_0x4eb5c1['BGmwW'](_0x2e727f,this,void 0x0,voi...",function(_0x209abc){return _0x3e5e47(setTimeou...
669,9984.0,https://snap.licdn.com/li.lms-analytics/insigh...,@0@32316,1.0,0.0,6.0,3925.0,17820.0,0.220258,4.540127,...,0.0,0.0,1.0,2.0,1.0,1.0,1.0,booking.com,return new(_0x42c4c6||(_0x42c4c6=Promise)),"_0x523f75=function(_0x24166a,_0x1b8b51,_0x42c4..."
670,9986.0,https://snap.licdn.com/li.lms-analytics/insigh...,@0@12097,1.0,0.0,2.0,3925.0,17820.0,0.220258,4.540127,...,0.0,0.0,1.0,2.0,1.0,1.0,1.0,booking.com,"function _0x318087(_0x5dbb61,_0xe317df,_0x1d4a...","function _0x2ba606(_0x4ca0c1,_0x35df86,_0x3a96..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1746,22027.0,https://static.ads-twitter.com/uwt.js,2735@0@34276,1.0,0.0,4.0,3925.0,17820.0,0.220258,4.540127,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,shopify.com,"m(o,e)","function(e,t){return e.__proto__=t,e},_(e,t)}f..."
1747,22031.0,https://static.ads-twitter.com/uwt.js,@0@47540,1.0,0.0,4.0,3925.0,17820.0,0.220258,4.540127,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,shopify.com,"d({customData:r,customParams:u,eventName:a,id:...","function Qe(t){var n="""";if(t.tagName===""IMG"")r..."
1748,22035.0,https://static.ads-twitter.com/uwt.js,@0@33973,1.0,0.0,4.0,3925.0,17820.0,0.220258,4.540127,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,shopify.com,"C(r,e.experimentId)","function(t){return this._invoke(e,n,t)})}n?o?o..."
1750,22043.0,https://static.ads-twitter.com/uwt.js,u@0@31050,1.0,0.0,2.0,3925.0,17820.0,0.220258,4.540127,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,shopify.com,"(e,{highFetchPriority:i,expv2:l})","function y(e,t,n,r,o,a,i){try{var l=e[a](i),s=..."


In [36]:
from dotenv import load_dotenv
load_dotenv()

True

In [38]:
# Use surrogate_2 to neutralize tracking method calls and add updated code to temp_2
from surrogate_2.llm_neutralize import neutralize_tracking_call, get_fallback_replacement

def neutralize_method_code(row, use_llm: bool = True, model: str = "gpt-4o"):
    """
    For a row with method_code (tracking call snippet), return a neutralized replacement
    that preserves return type but performs no tracking (no network/data exfil).
    """
    call_text = row["method_code"]
    if not call_text or not str(call_text).strip():
        return None
    context = call_text
    if pd.notna(row.get("method_definition")) and str(row["method_definition"]).strip():
        context = str(row["method_definition"])[:2000]
    if use_llm:
        replacement = neutralize_tracking_call(call_text, context, model=model)
    else:
        replacement = None
    if not replacement:
        replacement = get_fallback_replacement(call_text)
    return replacement

# Apply neutralization (use_llm=False to skip API and use heuristic fallback only)
USE_LLM = True  # Set to False if OPENAI_API_KEY is not set
temp_3 = temp_2.iloc[:20].assign(
    method_code_neutralized=temp_2.iloc[:20].apply(
        lambda row: neutralize_method_code(row, use_llm=USE_LLM),
        axis=1,
    )
)

In [41]:
temp_3[["method_definition", "method_code_neutralized"]].to_csv("temp_3.csv", index=False)

In [43]:
temp_3[["method_definition", "method_code_neutralized"]]

,method_definition,method_code_neutralized
654,"function _0x281135(_0x39f6b5,_0x2b0460,_0x26e0...",(function(){return true;})()
658,"_0x2e727f=function(_0x401c58,_0x20845b,_0x2b8f...",Promise.resolve(new Response())
664,function(_0x209abc){return _0x3e5e47(setTimeou...,(function(){return Promise.resolve();})()
669,"_0x523f75=function(_0x24166a,_0x1b8b51,_0x42c4...",return Promise.resolve(new Response())
670,"function _0x2ba606(_0x4ca0c1,_0x35df86,_0x3a96...",Promise.resolve()
678,function(){var _0x926722=_0x686ec6;return _0x4...,true
681,"function(_0x14b7b2,_0x5d7bd4){return _0x14b7b2...",(function(callback){callback();})(function(){i...
684,"_0x4ebcc6=function(_0x4360b3,_0x3b31f7,_0x4af9...",(function(){return true;})()
686,"function _0x53bbb5(_0x2d49a0,_0x4bb57b,_0xddb6...",Promise.resolve()
694,"_0x26b59a=function(_0x107f26,_0x3f51a3,_0x5c19...",Promise.resolve()
